# 08 — SBERT Baseline Evaluation

Evaluates the pre-trained Sentence-BERT model (all-MiniLM-L6-v2)
on IFRS taxonomy matching **before** fine-tuning, to establish a
baseline for comparison.

**Metrics**: Accuracy@1, Accuracy@5, MRR, per-category performance

In [ ]:
# Cell 1: Setup
!pip install -q sentence-transformers scikit-learn tqdm PyYAML

import json, os
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import classification_report
from tqdm.auto import tqdm
import yaml

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

CFG_PATH = Path('../configs/training_config.yaml')
if not CFG_PATH.exists():
    CFG_PATH = Path('/content/drive/MyDrive/numera-ml/configs/training_config.yaml')
cfg = yaml.safe_load(CFG_PATH.read_text())

DATA_DIR = Path(cfg['drive']['data_dir'])
scfg = cfg['sbert']

model = SentenceTransformer(scfg['model_name'])
print(f'Model: {scfg["model_name"]} loaded ✅')

In [ ]:
# Cell 2: Load IFRS taxonomy and test pairs

# Load taxonomy
taxonomy_file = Path('../../data/ifrs_taxonomy.json')
if not taxonomy_file.exists():
    taxonomy_file = DATA_DIR / 'ifrs_taxonomy.json'
with open(taxonomy_file) as f:
    taxonomy = json.load(f)

# Build taxonomy corpus: list of (code, label) pairs
taxonomy_items = []
if isinstance(taxonomy, dict) and 'elements' in taxonomy:
    for elem in taxonomy['elements']:
        taxonomy_items.append({
            'code': elem.get('code', elem.get('id', '')),
            'label': elem.get('label', elem.get('name', '')),
            'category': elem.get('category', 'unknown'),
        })
elif isinstance(taxonomy, list):
    for elem in taxonomy:
        taxonomy_items.append({
            'code': elem.get('code', ''),
            'label': elem.get('label', elem.get('name', '')),
            'category': elem.get('category', 'unknown'),
        })

# Load SBERT test pairs from notebook 03
pairs_file = DATA_DIR / 'labels' / 'sbert_training_pairs.json'
if pairs_file.exists():
    with open(pairs_file) as f:
        test_pairs = json.load(f)
else:
    # Fallback: generate synthetic test pairs from taxonomy
    test_pairs = [{'text': t['label'], 'taxonomy_code': t['code']} for t in taxonomy_items[:200]]

print(f'Taxonomy items: {len(taxonomy_items)}')
print(f'Test pairs: {len(test_pairs)}')

In [ ]:
# Cell 3: Encode taxonomy corpus
taxonomy_labels = [t['label'] for t in taxonomy_items]
taxonomy_codes = [t['code'] for t in taxonomy_items]

print('Encoding taxonomy corpus...')
taxonomy_embeddings = model.encode(taxonomy_labels, show_progress_bar=True, convert_to_tensor=True)
print(f'Encoded {len(taxonomy_labels)} taxonomy items, shape: {taxonomy_embeddings.shape}')

In [ ]:
# Cell 4: Evaluate retrieval accuracy

def evaluate_retrieval(model, test_pairs, taxonomy_embeddings, taxonomy_codes, k_values=[1, 3, 5, 10]):
    """Evaluate accuracy@k and MRR for taxonomy matching."""
    queries = [p['text'] for p in test_pairs]
    true_codes = [p['taxonomy_code'] for p in test_pairs]
    
    query_embeddings = model.encode(queries, show_progress_bar=True, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_embeddings, taxonomy_embeddings)
    
    results = {f'accuracy@{k}': 0 for k in k_values}
    mrr_sum = 0
    
    for i, (scores, true_code) in enumerate(zip(cos_scores, true_codes)):
        top_indices = torch.argsort(scores, descending=True)
        top_codes = [taxonomy_codes[idx] for idx in top_indices[:max(k_values)]]
        
        for k in k_values:
            if true_code in top_codes[:k]:
                results[f'accuracy@{k}'] += 1
        
        # MRR
        for rank, code in enumerate(top_codes, 1):
            if code == true_code:
                mrr_sum += 1.0 / rank
                break
    
    n = len(test_pairs)
    for k in k_values:
        results[f'accuracy@{k}'] /= n
    results['mrr'] = mrr_sum / n
    return results

import torch
baseline_metrics = evaluate_retrieval(model, test_pairs, taxonomy_embeddings, taxonomy_codes)

print('\n=== Baseline SBERT Results ===')
for metric, value in baseline_metrics.items():
    print(f'  {metric}: {value:.4f}')

In [ ]:
# Cell 5: Per-category breakdown
categories = set(p.get('namespace', 'unknown') for p in test_pairs)

print('\n=== Per-Category Accuracy@1 ===')
for category in sorted(categories):
    cat_pairs = [p for p in test_pairs if p.get('namespace', 'unknown') == category]
    if not cat_pairs:
        continue
    cat_metrics = evaluate_retrieval(model, cat_pairs, taxonomy_embeddings, taxonomy_codes, k_values=[1])
    print(f'  {category}: {cat_metrics["accuracy@1"]:.4f} ({len(cat_pairs)} pairs)')

# Save baseline results
results_file = DATA_DIR / 'sbert_baseline_results.json'
with open(results_file, 'w') as f:
    json.dump(baseline_metrics, f, indent=2)
print(f'\nSaved to: {results_file}')